# **STAGE 1 - ANONYMISE RAW DATA**

## Objectives

* I will first fetch my raw data - `animal_charity_donation_records.csv` - from Kaggle and save this in my `datasets/raw-data/` folder.
* Because this raw data contains **PII (Personally Identifiable Information)**, I will exclude this raw dataset from being uplaoded to my repository in GitHub.
* I will instead create an anonymised version of the raw data - this will be the dataset analysed and transformed in **Stage 2 - ETL**.

## Inputs

* The data input I need for this section is taken from Kaggle: `animal_charity_donation_records.csv`.
* It is a dataset containing **synthetic data**.
* It represents donations made to a fictitious wildlife charity.

## Outputs

* The output of **Stage 1 - Anonymise Raw Data** will be an anonymised version of the raw dataset, with columns containing PII either dropped or anonymised. 
* You will see at the end of this Jupyter Notebook that the columns `"name"` and `"email"` will be dropped from the DataFrame.
* The column `"donor_id"` will be anonymised using salting and hashing techniques. This column will then be dropped and replaced with one called `"anonymised_donor_id"`.

## Additional Comments

* I have separated this anonymisation stage from the ETL stage and placed in a separate Jupyter Notebook. The rationale behind this decision is that no code cells in this Jupyter Notebook will be able to be run because I am *excluding* the raw data from my GitHub upload. 
* The second section - **Stage 2 - ETL** - should be runnable because I will include the anonymised raw dataset in my repository. 
* In our course, we have not yet learned about ethical considerations when working with data, or what is best practice when it comes to dealing with PII in raw datasets. As a result, this section required a lot of research on my part to discover how best to anonymise data. I very much want to learn more when it comes to this practice and don't doubt there are areas for improvement in my methodology. 
* Despite working with a synthetic dataset, I wanted the opportunity to practice new skills (learning about **salting** and **hashing**) and to deliver anonymised donor IDs for this project.
* I have listed all my sources in the **Credits** section of my **README.md** file. I also used Claude generative AI to assist in the structure of my project - particularly in the context of the creation of a `.env` file and where that should sit in the folder/file hierarchy. 


---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'/Users/elliebrawn/Documents/vscode-projects/wildlife-charity-giving-analytics/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'/Users/elliebrawn/Documents/vscode-projects/wildlife-charity-giving-analytics'

---

# Import Libraries and Packages

In order to successfully read the raw dataset .csv file and then anonymise/ drop certain columns, I need to install the following libraries and packages:

In [4]:
import pandas as pd
import hashlib
from dotenv import load_dotenv

---

# Section 1 - Read Raw Data and Exclude from Repository Updates

* The first thing I did was to create a folder in datasets: `datasets/raw-data/`.
* I ensured the relative path was placed in the `.gitignore` file.
* This would ensure that the raw data and its PII would be excluded from uploads to GitHub.

---

# Section 2 - Load Raw Dataset and Identify Columns with PII

* Read the raw dataset - `animal_charity_donation_records.csv` - into a DataFrame and remove sensitive columns.
* Using `print(df.columns)` we ascertain that the columns `"donor_id"`, `"name"` and `"email"` need to be anonymised or removed.

In [5]:
# Load raw dataset
df = pd.read_csv("datasets/raw-data/animal_charity_donation_records.csv")

# Ascertain which columns contain sensitive data
print("All Columns in Raw Dataset:\n", df.columns)

All Columns in Raw Dataset:
 Index(['donor_id', 'age_group', 'gender', 'name', 'email', 'country',
       'donation_type', 'donation_amount', 'donation_date', 'payment_method',
       'newsletter_opt_in', 'referral_channel', 'sector', 'campaign'],
      dtype='object')


---

# Section 3 - Drop Columns with PII

* The next step is to use `df.drop` to remove the columns `"name"` and `"email"`from the DataFrame.

* There are two main reasons why I feel this is the correct next step:

1. As mentioned previosuly, I want to ensure I am creating a raw data file that includes no PII.
2. The nature of the analysis I want to do on this dataset means that specific names and email addresses will offer me little in terms of insights into donor giving behaviour.

In [6]:
# Drop "name" and "email" columns as they contain sensitive data
df = df.drop(columns=["name", "email"])

# Show the DataFrame after dropping columns
print("DataFrame Columns after dropping sensitive columns:\n", df.columns)

DataFrame Columns after dropping sensitive columns:
 Index(['donor_id', 'age_group', 'gender', 'country', 'donation_type',
       'donation_amount', 'donation_date', 'payment_method',
       'newsletter_opt_in', 'referral_channel', 'sector', 'campaign'],
      dtype='object')


---

# Section 4 - Anonymise Donor IDs

* As mentioned in the **Additional Comments** section above, the task of anonymising donor IDs involved learning a bit about salting and hashing.

## Salting
* The first thing I needed to learn was where you should put the salt (which would be added to each donor ID to complicate each ID before being hashed).
* I called this variable `DONOR_SALT`.

### Creating a `.env` Environment
* Having consulted blog posts/ articles on Medium.com and using generative AI, I discovered that this is often placed in a `.env` environment.
* In order to load the `DONOR_SALT` from the `.env`, I needed to load a new package - `python-dotenv==1.2.2`. This was added to my `requirements.txt` file.
* I was able to place into Claude my project structure and confirm where exactly the `.env` file was supposed to sit (I originally assumed that this would sit inside my `.venv`).
* I also made sure to exclude this `.env` from my commits byt putting it in `.gitignore` (this would ensure the `DONOR_SALT` was not shared) and was kep secure.

In [7]:
# Load variables from .env file into the project environment to be used
load_dotenv()

True

* When I ran this code cell above, the output was `True`, meaning that it has successfully loaded.

### Assign `DONOR_SALT` to a variable to use in the process of anonymising donor IDs
* Now that the `.env` had been loaded into the project, I needed to assign `DONOR_SALT` to a variable.
* I have selected a variable, `SALT`.

* I used **Microsoft Copilot's inline suggestions** to support me in writing these lines of code, particularly the first line because I hadn't used `os.getenv` previously.

In [8]:
# Assign the salt value from the environment variable to a variable
SALT = os.getenv("DONOR_SALT")

# Check that this loaded correctly by printing a message if SALT was loaded correctly
if SALT:
    print("The SALT variable has loaded correctly.")
else:
    print("The SALT variable has not loaded. Please check your .env file.")

The SALT variable has loaded correctly.


## Salting and Hashing


* Now to combine the donor ID values with the salt.
* I learned that environment variables in `.env` files are read as strings, therefore I need to make sure that the values in the `donor_id` colum are also strings.

In [9]:
# Check dtype of "donor_id" column
print(f"Data Type of donor_id column: {df['donor_id'].dtype}")

Data Type of donor_id column: object


* Because this code says that the datatype is object, I am unsure if *all* of the values are strings. I will therefore change the type of all the values in this column to string just to be sure (see next section).

### Create Function that Salts and Hashes values in `donor_id` column

* In the imported libraries and packages, I have already imported `hashlib` - within this library I am going to used SHA-256 to hash my donor_ids.
* In order to do this, I learned that the string we create made up of the donor_id plus the SALT *must be coverted to bytes*.

In [10]:
# Create a function that we .apply to the values of the donor_id column to combine with SALT and hash the result
def hash_donor_id(donor_id):
    # Combine the donor_id and SALT into one string and conver it to bytes
    salted_donor_id = (str(donor_id) + SALT).encode("utf-8")
    # Hash the salted donor_id using SHA-256
    hashed_donor_id = hashlib.sha256(salted_donor_id).hexdigest()
    return hashed_donor_id

* To test that the function worked as expected, I created a test where I call the newly-created function. 

In [11]:
hash_donor_id("donor42")  # Example usage of the function

'a7d15a3c652843511cd5c4f55eb8394b7cf3855216f85196063b735aca42e9ad'

* Because the output is a hexadecimal string with character 0-9 and a-f, I can be confident that this function has worked as expected.

---

# Section 5 - Update DataFrame with Anonymised Donor IDs only

* I created a new anonymised column in the DataFrame and checked its contents to ensure that the function has worked as expected on the column.

In [12]:
# Create anonymised column in the raw DataFrame
df["anonymised_donor_id"] = df["donor_id"].apply(hash_donor_id)

# Check that the new column has been created and the new values have been hashed
print(f"Contents of 'anonymised_donor_id' column:\n{df["anonymised_donor_id"].head(5)}")

Contents of 'anonymised_donor_id' column:
0    f887acec7d236a724e0cd0df5112c6a7641e2b6867aeb9...
1    db1fd2f0b9dd06646c1713945fda992a7250b512cce720...
2    2723050e7f8594d0642ce8eb7396f17628b99c2ac9a038...
3    250222dcc99547fe972a1ee5702b4ab7d96246329228cb...
4    e5b077c046e41a98c2c7689f06f41bc7a43ca2667a1968...
Name: anonymised_donor_id, dtype: object


* Now that we have the new anonymised version of the donor IDs, I am safe to delete the original `donor_id` column from my DataFrame.

In [13]:
# Drop the "donor_id" column from the DataFrame as we did already with "name" and "email"
df = df.drop(labels=["donor_id"], axis=1)

# Confirm the column deletion by printing the DataFrame columns
print("DataFrame Columns after dropping 'donor_id' column:\n", df.columns)

DataFrame Columns after dropping 'donor_id' column:
 Index(['age_group', 'gender', 'country', 'donation_type', 'donation_amount',
       'donation_date', 'payment_method', 'newsletter_opt_in',
       'referral_channel', 'sector', 'campaign', 'anonymised_donor_id'],
      dtype='object')


* Finally, to clean up my DataFrame, I moved the "anonymised_donor_id" column from the end of the DataFrame to the beginning. 
* This was a new skill that I had not learned before, so I relied on an article written via Medium.com to get the correct code. The author and the article are listed in the **Credits** section of the **README.md**.

In [14]:
# Update column order to have the "anonymised_donor_id" column first, followed by the other columns
new_column_order = ["anonymised_donor_id"] + [col for col in df.columns if col != "anonymised_donor_id"]
df = df[new_column_order]
df.head(20)

,anonymised_donor_id,age_group,gender,country,donation_type,donation_amount,donation_date,payment_method,newsletter_opt_in,referral_channel,sector,campaign
0,f887acec7d236a724e0cd0df5112c6a7641e2b6867aeb9...,50-65,Female,UK,Monthly,115.31,2024-10-24,Paypal,False,Website,Real Estate,Rescue Orphaned Gorillas
1,db1fd2f0b9dd06646c1713945fda992a7250b512cce720...,50-65,Female,USA,One-time,8.60,2024-06-21,Bank Transfer,False,Online advertising,Logistics,Rescue Orphaned Gorillas
2,2723050e7f8594d0642ce8eb7396f17628b99c2ac9a038...,50-65,Female,USA,One-time,40.07,2024-08-21,Bank Transfer,False,Online advertising,Media & Communication,Marine Mammal Defense Mission
3,250222dcc99547fe972a1ee5702b4ab7d96246329228cb...,18-29,Male,USA,One-time,45.17,2023-10-09,Bank Transfer,False,Online advertising,Government,Wildlife Rescue Van Drive
4,e5b077c046e41a98c2c7689f06f41bc7a43ca2667a1968...,30-49,Female,UK,One-time,85.84,2024-09-01,Bank Transfer,True,Newsletter,Science & Research,Habitat for Hope
5,a112b48e7e42a139743ef4a31a814b1ebbc6ff4869e720...,50-65,Male,Canada,One-time,293.22,2024-07-25,Cheque,True,Online advertising,Pharmaceuticals,Habitat for Hope
6,5e139393e3f3a2bff6d40e449f2f19781c33c6f1ec45ba...,18-29,Female,Netherlands,One-time,15.00,2023-07-21,Paypal,True,Newsletter,Science & Research,Habitat for Hope
7,612ed0d31f700426f6ac6f8bf0496c39e74f6e6218ad6c...,50-65,Female,France,One-time,93.42,2024-01-07,Bank Transfer,False,Newsletter,Government,Wildlife Rescue Van Drive
8,020af8af45b8ae11ffb5d12d3301bdeacde2e5a3b61962...,50-65,Female,Netherlands,Monthly,19.46,2025-05-02,Bank Transfer,True,Word of mouth,Agriculture,Defend the Great Apes
9,6ab3d5603d6b1abff011a2d1f36f188a4359bbd54b3d93...,50-65,Female,USA,Monthly,209.59,2024-03-21,Credit Card,True,Website,Hospitality,Defend the Great Apes


---

# Section 6 - Save Anonymised DataFrame content to a CSV file

* The final step is to save the newly-created anonymised DataFrame into a CSV file that we can use for our ETL section without fear of showing any PII
* I will name this file `animal_charity_donation_records_anonymised.csv`

In [15]:
df.to_csv("datasets/anonymised-raw-data/animal_charity_donation_records_anonymised.csv", index=False)

# Conclusion